# FinGuard Model Training in Google Colab
This notebook contains the complete pipeline to train the FinGuard Fraud Detection model using Google Colab's cloud resources.

**Instructions:**
1. Run the first cell and click **Choose Files** to upload your local `transactions.csv` dataset.
2. Run the remaining cells sequentially to preprocess data, train the model, evaluate it, and generate SHAP explanations.
3. The final cell will automatically zip and download the trained model and scaler back to your computer.

In [ ]:
!pip install shap imbalanced-learn lightgbm joblib

import os
from google.colab import files

print("Please select and upload your local 'transactions.csv' file:")
uploaded = files.upload()

# Move the uploaded file to a raw directory structure
os.makedirs("data/raw", exist_ok=True)
if "transactions.csv" in uploaded:
    os.rename("transactions.csv", "data/raw/transactions.csv")
    print("Dataset successfully uploaded and moved to data/raw/transactions.csv!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import joblib

# Load raw dataset
df = pd.read_csv("data/raw/transactions.csv")

# Create engineered features
df['errorBalanceOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']
df['errorBalanceDest'] = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']
df['is_merchant_dest'] = df['nameDest'].str.startswith('M').astype(int)

# One-hot encode the categorical 'type' column
type_dummies = pd.get_dummies(df['type'], prefix='type', dtype=int)
df = pd.concat([df, type_dummies], axis=1)

# Drop identifiers
columns_to_drop = ['nameOrig', 'nameDest', 'type', 'isFlaggedFraud']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Split data (80% Train, 20% Test)
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale continuous numerical features
cols_to_scale = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
scaler = RobustScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# Save the fitted scaler weights
os.makedirs("models/trained", exist_ok=True)
joblib.dump(scaler, "models/trained/scaler.pkl")
print("Scaler fitted and saved successfully.")

In [ ]:
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

print("Applying SMOTE oversampling to training split...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Resampled Training size: {X_train_res.shape}")

print("Training LightGBM model on balanced dataset...")
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    verbosity=-1
)
lgbm_model.fit(X_train_res, y_train_res)

# Save the trained model weights
joblib.dump(lgbm_model, "models/trained/lgbm_model.pkl")
print("Model trained and saved successfully.")

In [ ]:
from sklearn.metrics import classification_report, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Predict test labels and risk probabilities
preds = lgbm_model.predict(X_test)
probs = lgbm_model.predict_proba(X_test)[:, 1]

print("Model Evaluation Metrics:")
print(classification_report(y_test, preds))

# Calculate PR-AUC
precision, recall, _ = precision_recall_curve(y_test, probs)
pr_auc = auc(recall, precision)
print(f"Precision-Recall AUC: {pr_auc:.4f}")

# Plot PR-Curve
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f'LightGBM (AUC = {pr_auc:.3f})', color='#6366f1')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import shap

# Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(lgbm_model)

# Compute global SHAP values for 200 random test samples
X_sample = X_test.sample(n=200, random_state=42)
shap_values = explainer(X_sample)

# Plot Global SHAP values
print("Global Feature Importance (SHAP Plot):")
if len(shap_values.values.shape) == 3:
    shap.summary_plot(shap_values.values[:, :, 1], X_sample)
else:
    shap.summary_plot(shap_values, X_sample)

In [ ]:
from google.colab import files

print("Zipping weights for download...")
!zip -r model_weights.zip models/trained/

print("Downloading model_weights.zip...")
files.download("model_weights.zip")